# Lab 7c: Model selection and the three-way split

We will work with the Wisconsin Diagnostic Breast Cancer data, which includes 569 tumors with  30 measurements each. Each tumor is
labelled malignant or benign. It is a real diagnostic dataset,
collected at the University of Wisconsin and published on the UCI repository.

In this lab we want to select a model, and also select a best learning rate, regularization, etc. How do we do this without introducing bias?
We do this by dividing the data into three parts rather than
two:

| set | rows | what it is for |
|---|---|---|
| **training** | 60% | fitting $\mathbf{w}$ and $b$ |
| **validation** | 20% | model, preprocessing, hyperparameters |
| **test** | 20% | estimate of generalizability at the end |



## Setup

Import the data with:

```python
from sklearn.datasets import load_breast_cancer
```

If you would rather parse the raw file, it is at
`https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data`,
where column 1 holds `M` or `B` and columns 2 to 31 hold the measurements.


In [1]:
# PROVIDED
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import confusion_matrix, recall_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target

print(f'{X.shape[0]} tumors, {X.shape[1]} measurements each')
print(f'labels: 0 = {data.target_names[0]}, 1 = {data.target_names[1]}')
print(f'counts: {np.bincount(y)}')
print()
print('the first five measurements are called:')
for name in data.feature_names[:5]:
    print(f'   {name}')

569 tumors, 30 measurements each
labels: 0 = malignant, 1 = benign
counts: [212 357]

the first five measurements are called:
   mean radius
   mean texture
   mean perimeter
   mean area
   mean smoothness


In [3]:
X.shape

(569, 30)

## Question 1: build the split

Divide the data into training, validation and test sets, 60 / 20 / 20, with the
class balance preserved in all three. Use stratify and random_state = 0.

You should end up with **341 / 114 / 114 rows**. 

Then find the correct baseline accuracy we should compare our results against.

In [5]:
X_train, X_tmp, y_train, y_tmp = train_test_split( 
    X, y, test_size=0.4, stratify=y, random_state=0)  # 60/40 split
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=0)  # 20/20 split 

In [8]:
print('train:', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)

train: (341, 30) val (114, 30) test (114, 30)


In [13]:
print(100*np.mean(y_val), 'percent is benign')

63.1578947368421 percent is benign


*Your answer here.*

## Question 2: choose a classifier, on the validation set

* `Perceptron`
* `LogisticRegression`
* `LinearSVC`

Train each model with and without the standard scaler. Report **validation** accuracy for all six, and
pick a winner.

In [15]:
perceptron = Perceptron(random_state=0)
logreg = LogisticRegression(max_iter=5000)
linearsvm = LinearSVC(C=1.0, max_iter=20000, random_state=0)

models = [('Perceptron', perceptron),
          ('Logistic Regression', logreg), 
          ('Linear SVM', linearsvm)]

for name, model in models:
    # raw data
    fit_model = model.fit(X_train, y_train)
    raw_score = fit_model.score(X_val, y_val)
    # with standard scaling
    fit_model_scaled = make_pipeline(StandardScaler(), model).fit(X_train, y_train)
    scaled_score = fit_model_scaled.score(X_val, y_val)

    print(f'{name}, raw:{raw_score:f}, scaled: {scaled_score:f}, diff: {scaled_score - raw_score:.4f}')

Perceptron, raw:0.894737, scaled: 0.956140, diff: 0.0614
Logistic Regression, raw:0.929825, scaled: 0.947368, diff: 0.0175
Linear SVM, raw:0.938596, scaled: 0.947368, diff: 0.0088


*Your answer here.*

## Question 3: tune the regularization strength, on the validation set

Take `LogisticRegression` with a `StandardScaler` and sweep `C` across several
orders of magnitude. Something like
`[1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1e4]` is a reasonable grid.

For each value report three things: **training** accuracy, **validation**
accuracy, and the length of the weight vector, `np.linalg.norm(model[-1].coef_)`.
Then plot the training and validation curves against $C$ on a log scale.

Recall that in scikit-learn $C$ is an inverse penalty. Small $C$ means heavy
regularization. The objective is

$$\min_{\mathbf{w}, b} \; \tfrac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \ell\big(y^{(i)}, \mathbf{w}^\top\mathbf{x}^{(i)} + b\big)$$

so as $C$ grows the penalty on $\|\mathbf{w}\|$ stops mattering and the fit
chases the training data.

In [18]:
# Your code here.
grid = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1e4]

train_accs = []
val_accs = []
norms = []

for C in grid:
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X_train, y_train)
    train_accs.append(pipe.score(X_train, y_train))
    val_accs.append(pipe.score(X_val, y_val))
    norms.append( np.linalg.norm(pipe[-1].coef_) )
    print(f'{C}, {train_accs[-1]}, {val_accs[-1]}, {norms[-1]}')

0.0001, 0.6304985337243402, 0.631578947368421, 0.044179095438302556
0.001, 0.873900293255132, 0.8947368421052632, 0.2512396472564405
0.01, 0.9501466275659824, 0.956140350877193, 0.7416723311072498
0.1, 0.9853372434017595, 0.956140350877193, 1.6636361378805882
1, 0.9912023460410557, 0.9473684210526315, 3.3536910496427446
10, 0.9912023460410557, 0.9473684210526315, 8.335954758147086
100, 0.9970674486803519, 0.9298245614035088, 24.06270278044235
10000.0, 1.0, 0.9122807017543859, 115.3627274013807


## Question 3b: Do one more test over some hyperparameter.

This can be of your choice. For example, search over C in linear SVM, try a different regularization, play around with max_iterations...

## Question 4: test

Finally, evaluate your best model/hyperparameter on the test set. Write what you expected, and what you found.

In [ ]:
# Your code here.

*Your answer here.*

## Question 5: cross-validation


Rerun the $C$ sweep from Question 3 using `cross_val_score` with
`StratifiedKFold(5, shuffle=True, random_state=0)` on the combined training and
validation rows. Report the cross-validated mean and standard deviation beside
the single-split validation number for each $C$.


We will discuss this in class on 11/4.

In [ ]:
# Your code here.

*Your answer here.*

## Question 6: Is accuracy an adequate measure?

Compute the confusion matrix.

We will discuss this in class 11/4.

In [ ]:
# Your code here.

*Your answer here.*